# JEE/NEET figure extraction (v4)
Run top to bottom. Sections: 0 setup · 1 config · 2 verify refs · 3 ground truth · 4 review · 5 held-out eval · 6 bulk (unmarked) · 7 export.
Everything is resumable; outputs go to a **new versioned folder** and existing `recovered_crops*` are never touched.

In [ ]:
!pip -q install "pymupdf>=1.24" opencv-python-headless pytest 2>&1 | tail -1
import pymupdf, cv2, PIL, sys, shutil, os
print(pymupdf.__doc__.split(chr(10))[0], cv2.__version__, PIL.__version__, 'RAM/disk:', shutil.disk_usage('/content').free//2**30, 'GB free')
!nvidia-smi -L 2>&1 | head -1  # GPU is NOT needed for this pipeline

In [ ]:
!mkdir -p /content/figx

In [ ]:
%%writefile /content/figx/__init__.py

In [ ]:
%%writefile /content/figx/core.py
"""Core geometry, rendering, annotation-box recovery and crop QA.

Coordinates: all boxes crossing module boundaries are in PDF points of the
page's visible rect (page.rect, rotation already applied), as (x0, y0, x1, y1).
Pixel boxes are only used inside a single rendered image.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from pathlib import Path

import cv2
import numpy as np
import pymupdf

Box = tuple[float, float, float, float]


# ----------------------------------------------------------------- geometry
def area(b: Box) -> float:
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def intersection(a: Box, b: Box) -> float:
    return area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))


def iou(a: Box, b: Box) -> float:
    i = intersection(a, b)
    u = area(a) + area(b) - i
    return i / u if u > 0 else 0.0


def containment(inner: Box, outer: Box) -> float:
    """Fraction of `inner` that lies inside `outer`."""
    a = area(inner)
    return intersection(inner, outer) / a if a > 0 else 0.0


def union(a: Box, b: Box) -> Box:
    return (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))


def expand(b: Box, d: float) -> Box:
    return (b[0] - d, b[1] - d, b[2] + d, b[3] + d)


def clip(b: Box, w: float, h: float) -> Box:
    return (max(0.0, b[0]), max(0.0, b[1]), min(w, b[2]), min(h, b[3]))


def suppress_duplicates(boxes: list[Box], scores: list[float] | None = None,
                        iou_thr: float = 0.85, contain_thr: float = 0.95) -> list[int]:
    """Keep-indices after removing near-identical boxes.

    Only boxes that are essentially the same region (IoU >= iou_thr, or one
    almost entirely inside the other AND similar in size) are merged, so two
    distinct neighbouring targets that merely overlap both survive.
    """
    scores = scores or [area(b) for b in boxes]
    order = sorted(range(len(boxes)), key=lambda i: -scores[i])
    keep: list[int] = []
    for i in order:
        dup = False
        for k in keep:
            a, b = boxes[i], boxes[k]
            size_ratio = min(area(a), area(b)) / max(area(a), area(b), 1e-9)
            if iou(a, b) >= iou_thr or (
                max(containment(a, b), containment(b, a)) >= contain_thr and size_ratio >= 0.6
            ):
                dup = True
                break
        if not dup:
            keep.append(i)
    return sorted(keep)


# ---------------------------------------------------------------- rendering
def render(page: pymupdf.Page, dpi: float | None = None, size: tuple[int, int] | None = None,
           clip_box: Box | None = None) -> np.ndarray:
    """Render to BGR uint8. Either dpi, or an exact (w, h) pixel size for the full page."""
    r = page.rect
    if size is not None:
        mat = pymupdf.Matrix(size[0] / r.width, size[1] / r.height)
    else:
        z = (dpi or 150) / 72.0
        mat = pymupdf.Matrix(z, z)
    pm = page.get_pixmap(matrix=mat, alpha=False, colorspace=pymupdf.csRGB,
                         clip=pymupdf.Rect(clip_box) if clip_box else None)
    img = np.frombuffer(pm.samples, np.uint8).reshape(pm.h, pm.w, 3)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)


# ------------------------------------------------------------ red detection
@dataclass
class RedConfig:
    hue_lo: int = 10        # hue <= hue_lo  (OpenCV hue 0..179)
    hue_hi: int = 165       # or hue >= hue_hi  (covers red and maroon/crimson)
    sat_min: int = 80
    val_min: int = 50       # maroon is dark; keep low
    analysis_dpi: int = 100
    min_side_pt: float = 18.0
    min_side_support: float = 0.55   # each side must be >=55% red along its length
    max_interior_fill: float = 0.30  # solid blobs (cartoon clothing) fail this
    max_thickness_frac: float = 0.25


def red_mask(bgr: np.ndarray, cfg: RedConfig = RedConfig()) -> np.ndarray:
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    m = ((h <= cfg.hue_lo) | (h >= cfg.hue_hi)) & (s >= cfg.sat_min) & (v >= cfg.val_min)
    return m.astype(np.uint8)


@dataclass
class FrameCheck:
    ok: bool
    side_support: tuple[float, float, float, float]   # top, bottom, left, right
    interior_fill: float
    thickness_px: int
    reason: str = ""


def verify_frame(mask: np.ndarray, x: int, y: int, w: int, h: int,
                 cfg: RedConfig = RedConfig()) -> FrameCheck:
    """Is the red component inside bbox (x,y,w,h) a rectangular *frame*?

    Signals: four-side support (each border band is mostly red), low interior
    fill (frames are hollow), and a plausible stroke thickness. Rounded corners
    are tolerated because support is measured on the middle 70% of each side.
    """
    sub = mask[y:y + h, x:x + w]
    # thickness: median red run length entering from each side along middle rows/cols
    def run(line: np.ndarray) -> int:
        nz = np.flatnonzero(line == 0)
        return int(nz[0]) if nz.size else len(line)
    mids_r = range(int(h * 0.3), max(int(h * 0.7), int(h * 0.3) + 1))
    mids_c = range(int(w * 0.3), max(int(w * 0.7), int(w * 0.3) + 1))
    runs = [run(sub[r, :]) for r in mids_r] + [run(sub[r, ::-1]) for r in mids_r] + \
           [run(sub[:, c]) for c in mids_c] + [run(sub[::-1, c]) for c in mids_c]
    t = max(1, int(np.median(runs))) if runs else 1
    band = max(t + 2, 3)
    cx0, cx1 = int(w * 0.15), max(int(w * 0.85), int(w * 0.15) + 1)
    cy0, cy1 = int(h * 0.15), max(int(h * 0.85), int(h * 0.15) + 1)
    sup = (
        float(sub[:band, cx0:cx1].max(0).mean()),
        float(sub[h - band:, cx0:cx1].max(0).mean()),
        float(sub[cy0:cy1, :band].max(1).mean()),
        float(sub[cy0:cy1, w - band:].max(1).mean()),
    )
    inner = sub[band + 1:h - band - 1, band + 1:w - band - 1]
    fill = float(inner.mean()) if inner.size else 1.0
    reason = ""
    if min(sup) < cfg.min_side_support:
        reason = f"weak_side_support({min(sup):.2f})"
    elif fill > cfg.max_interior_fill:
        reason = f"solid_interior({fill:.2f})"
    elif t > cfg.max_thickness_frac * min(w, h):
        reason = f"too_thick({t}px)"
    return FrameCheck(not reason, sup, fill, t, reason)


@dataclass
class Annotation:
    page: int                 # 0-based
    box_pt: Box               # outer frame, marked-page points
    inner_pt: Box             # frame interior (border removed), marked-page points
    thickness_pt: float
    method: str               # "diff" | "single"
    check: dict = field(default_factory=dict)
    accepted: bool = True
    reason: str = ""


def _components_to_annotations(mask: np.ndarray, page_idx: int, page_rect: pymupdf.Rect,
                               cfg: RedConfig, method: str) -> list[Annotation]:
    H, W = mask.shape
    sx, sy = page_rect.width / W, page_rect.height / H
    # close small gaps in hand-drawn/compressed borders so a frame is one component
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(closed, 8)
    out: list[Annotation] = []
    min_px = cfg.min_side_pt / sx
    for k in range(1, n):
        x, y, w, h, _ = (int(v) for v in stats[k])
        if w < min_px or h < min_px:
            continue
        chk = verify_frame(closed, x, y, w, h, cfg)
        t = chk.thickness_px + 1
        box = (x * sx, y * sy, (x + w) * sx, (y + h) * sy)
        inner = ((x + t) * sx, (y + t) * sy, (x + w - t) * sx, (y + h - t) * sy)
        out.append(Annotation(page_idx, box, inner, t * sx, method, asdict(chk), chk.ok, chk.reason))
    acc = [a for a in out if a.accepted]
    keep = set(suppress_duplicates([a.box_pt for a in acc]))
    for i, a in enumerate(acc):
        if i not in keep:
            a.accepted, a.reason = False, "duplicate"
    return out


def recover_annotations(marked: pymupdf.Page, original: pymupdf.Page | None,
                        cfg: RedConfig = RedConfig()) -> tuple[list[Annotation], dict]:
    """Find human-drawn red/maroon frames on a marked page.

    With the unmarked original available ("diff" mode) only red pixels that are
    NEW in the marked page are considered, so red content that already exists in
    the figure (cartoon clothing, red graph lines) cannot become a box. Without
    an original ("single" mode) frame verification alone must reject blobs.
    Returns (annotations, diagnostics) where diagnostics includes the measured
    alignment shift between the two renders.
    """
    W = int(round(marked.rect.width * cfg.analysis_dpi / 72))
    H = int(round(marked.rect.height * cfg.analysis_dpi / 72))
    m_img = render(marked, size=(W, H))
    rm = red_mask(m_img, cfg)
    diag: dict = {"red_px_marked": int(rm.sum())}
    if original is not None:
        o_img = render(original, size=(W, H))
        g1 = cv2.cvtColor(m_img, cv2.COLOR_BGR2GRAY).astype(np.float32)
        g2 = cv2.cvtColor(o_img, cv2.COLOR_BGR2GRAY).astype(np.float32)
        (dx, dy), resp = cv2.phaseCorrelate(g2, g1)
        diag.update(shift_px=(round(dx, 2), round(dy, 2)), shift_response=round(resp, 3))
        if abs(dx) > 1 or abs(dy) > 1:   # re-register original onto marked before diffing
            o_img = cv2.warpAffine(o_img, np.float32([[1, 0, dx], [0, 1, dy]]), (W, H),
                                   borderValue=(255, 255, 255))
        ro = cv2.dilate(red_mask(o_img, cfg), np.ones((5, 5), np.uint8))
        rm = rm & (1 - ro)
        diag["red_px_added"] = int(rm.sum())
        method = "diff"
    else:
        method = "single"
    return _components_to_annotations(rm, marked.number, marked.rect, cfg, method), diag


def map_marked_to_original(box: Box, marked: pymupdf.Page, original: pymupdf.Page,
                           shift_px: tuple[float, float] = (0, 0), dpi: float = 100) -> Box:
    """Map a marked-page point box to the original page (scale + measured shift)."""
    sx = original.rect.width / marked.rect.width
    sy = original.rect.height / marked.rect.height
    dx, dy = -shift_px[0] * 72 / dpi, -shift_px[1] * 72 / dpi
    return (box[0] * sx + dx, box[1] * sy + dy, box[2] * sx + dx, box[3] * sy + dy)


# --------------------------------------------------------------- crop + QA
def tighten(img: np.ndarray, pad: int = 6, white: int = 235) -> tuple[np.ndarray, tuple[int, int, int, int]]:
    """Trim surrounding whitespace, keep `pad` px margin. Returns crop and pixel bbox."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ys, xs = np.nonzero(g < white)
    if xs.size == 0:
        return img, (0, 0, img.shape[1], img.shape[0])
    x0, x1 = max(0, xs.min() - pad), min(img.shape[1], xs.max() + 1 + pad)
    y0, y1 = max(0, ys.min() - pad), min(img.shape[0], ys.max() + 1 + pad)
    return img[y0:y1, x0:x1], (int(x0), int(y0), int(x1), int(y1))


def quality_flags(img: np.ndarray, page: pymupdf.Page, box_pt: Box, cfg: RedConfig = RedConfig()) -> dict:
    """Heuristic review flags. They guide human review; they never delete crops."""
    h, w = img.shape[:2]
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ink = float((g < 200).mean())
    edge = 3
    edge_ink = max(float((g[:edge] < 160).mean()), float((g[-edge:] < 160).mean()),
                   float((g[:, :edge] < 160).mean()), float((g[:, -edge:] < 160).mean()))
    rm = red_mask(img, cfg)
    bw = max(2, int(0.06 * min(h, w)))       # annotation remnants live at the crop border
    red_left = max(float(rm[:bw].mean()), float(rm[-bw:].mean()),
                   float(rm[:, :bw].mean()), float(rm[:, -bw:].mean()))
    # text contamination: fraction of crop area covered by text words that are part of long lines
    rect = pymupdf.Rect(box_pt)
    words = page.get_text("words", clip=rect)
    word_area = sum(area((x0, y0, x1, y1)) for x0, y0, x1, y1, *_ in words)
    text_frac = word_area / max(area(box_pt), 1e-9)
    flags = []
    if ink < 0.004:
        flags.append("empty")
    if min(h, w) < 40 or max(h, w) / max(min(h, w), 1) > 12:
        flags.append("tiny_or_narrow")
    if edge_ink > 0.35:
        flags.append("content_touches_edge")
    if red_left > 0.15:
        flags.append("red_border_remains")
    if text_frac > 0.35:
        flags.append("text_heavy")
    if ink < 0.02 and min(h, w) > 200:
        flags.append("mostly_whitespace")
    return {"ink": round(ink, 4), "edge_ink": round(edge_ink, 3), "red_frac": round(red_left, 4),
            "text_frac": round(text_frac, 3), "words": len(words), "flags": flags}


def crop_region(page: pymupdf.Page, box_pt: Box, dpi: int = 300, tight: bool = True) -> tuple[np.ndarray, Box]:
    """Render a page region; optionally trim whitespace. Returns image and final point box."""
    box_pt = clip(box_pt, page.rect.width, page.rect.height)
    img = render(page, dpi=dpi, clip_box=box_pt)
    if not tight:
        return img, box_pt
    img2, (x0, y0, x1, y1) = tighten(img)
    s = 72.0 / dpi
    return img2, (box_pt[0] + x0 * s, box_pt[1] + y0 * s, box_pt[0] + x1 * s, box_pt[1] + y1 * s)


def save_png(img: np.ndarray, path: Path, dpi: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(".png", img)
    if not ok:
        raise IOError(f"encode failed: {path}")
    tmp = path.with_suffix(".tmp")
    tmp.write_bytes(buf.tobytes())
    tmp.replace(path)

In [ ]:
%%writefile /content/figx/detect.py
"""Figure detection on UNMARKED pages (no human boxes).

PDF-native strategy: figures in these papers are embedded raster images and/or
vector drawings. We collect graphic elements, drop document boilerplate
(logos, barcodes, header rules repeated across pages), drop full-page scans
and table grids, cluster what remains, and attach short label text (axis
names, "10 m/s") that sits inside or right next to a cluster.

Scanned pages (one full-page image) fall back to a raster detector that
removes text-sized connected components and keeps large ink blobs; those
crops are always flagged for review.
"""
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass

import cv2
import numpy as np
import pymupdf

from .core import Box, area, containment, expand, iou, render, union


@dataclass
class DetectConfig:
    min_w_pt: float = 30.0
    min_h_pt: float = 22.0
    min_area_pt2: float = 1500.0
    max_page_frac: float = 0.60          # bigger than this = page scan / background
    merge_gap_pt: float = 9.0            # elements closer than this join one figure
    boilerplate_page_frac: float = 0.3   # element repeated on >=30% of pages = boilerplate
    margin_band_frac: float = 0.07       # header/footer band height as page fraction
    label_reach_pt: float = 14.0         # how far outside a cluster a label may sit
    label_max_chars: int = 18
    skip_tables: bool = True
    raster_fallback: bool = True
    max_aspect: float = 9.0              # wider/taller than this = banner/rule strip
    thin_pt: float = 2.5                 # stroke thickness of a "line" element
    scanned_doc_frac: float = 0.5        # raster fallback only if most pages are scans
    inline_max_h_pt: float = 36.0        # short images with text beside them are inline math


def _key(b: Box) -> tuple:
    return tuple(int(round(v / 3)) for v in b)   # 3pt tolerance


def boilerplate_keys(doc: pymupdf.Document, cfg: DetectConfig) -> set:
    """Graphic boxes that recur at the same position on many pages (logos, rules, barcodes)."""
    cnt: Counter = Counter()
    for p in doc:
        H = p.rect.height
        zone = lambda b: b[3] <= 0.15 * H or b[1] >= 0.85 * H   # header/footer zones only
        seen = set()
        for info in p.get_image_info():
            if zone(info["bbox"]):
                seen.add(_key(tuple(info["bbox"])))
        for d in p.get_drawings():
            if zone(d["rect"]):
                seen.add(_key(tuple(d["rect"])))
        cnt.update(seen)
    thr = max(3, int(cfg.boilerplate_page_frac * len(doc)))
    return {k for k, c in cnt.items() if c >= thr}


def _kind(b: Box, cfg: DetectConfig) -> str:
    w, h = b[2] - b[0], b[3] - b[1]
    if h <= cfg.thin_pt and w > cfg.thin_pt:
        return "hline"
    if w <= cfg.thin_pt and h > cfg.thin_pt:
        return "vline"
    return "shape"


def table_regions(lines: list[Box], page_w: float, page_h: float, cfg: DetectConfig) -> list[Box]:
    """Ruled tables from the network of thin horizontal/vertical strokes.

    Tables here are often drawn cell by cell, so strokes are rasterised at
    1px/pt and connected; a network with >=3 distinct horizontal rows and >=3
    distinct vertical columns is a table.
    """
    W, H = int(page_w) + 2, int(page_h) + 2
    m = np.zeros((H, W), np.uint8)
    for b in lines:
        cv2.rectangle(m, (int(b[0]), int(b[1])), (int(np.ceil(b[2])), int(np.ceil(b[3]))), 1, -1)
    n, lab, st, _ = cv2.connectedComponentsWithStats(cv2.dilate(m, np.ones((3, 3), np.uint8)), 8)
    out = []
    for k in range(1, n):
        x, y, w, h, _ = st[k]
        comp = [b for b in lines if lab[min(int(b[1]) + 1, H - 1), min(int(b[0]) + 1, W - 1)] == k]
        rows = {round(b[1] / 3) for b in comp if _kind(b, cfg) == "hline"}
        cols = {round(b[0] / 3) for b in comp if _kind(b, cfg) == "vline"}
        if len(rows) >= 3 and len(cols) >= 3 and w >= 40 and h >= 30:
            out.append((x - 1.5, y - 1.5, x + w + 1.5, y + h + 1.5))
    return out


def _graphic_elements(page: pymupdf.Page, boiler: set, cfg: DetectConfig) -> tuple[list[Box], bool]:
    W, H = page.rect.width, page.rect.height
    pa = W * H
    band = cfg.margin_band_frac * H
    els: list[Box] = []
    scanned = False
    words = page.get_text("words")
    for info in page.get_image_info():
        b = tuple(info["bbox"])
        if area(b) >= cfg.max_page_frac * pa:
            scanned = True
            continue
        if _key(b) in boiler or b[3] <= band or b[1] >= H - band:
            continue
        if _is_inline(b, words, cfg):
            continue
        els.append(b)
    for d in page.get_drawings():
        b = tuple(d["rect"])
        w, h = b[2] - b[0], b[3] - b[1]
        if _key(b) in boiler or b[3] <= band or b[1] >= H - band:
            continue
        if area(b) >= cfg.max_page_frac * pa:
            continue
        if w > 0.75 * W and h < 4:          # full-width separator rule
            continue
        if d.get("fill") is not None and w > 0.5 * W and h > 0.5 * H:   # page background
            continue
        if h <= 12 and w <= 80 and len(d["items"]) > 20:   # text drawn as vector outlines
            continue
        els.append(b)
    return els, scanned


def _is_inline(b: Box, words: list, cfg: DetectConfig) -> bool:
    """Short image sitting on a text line with words immediately left/right of it."""
    if b[3] - b[1] > cfg.inline_max_h_pt:
        return False
    for w in words:
        cy = (w[1] + w[3]) / 2
        if b[1] <= cy <= b[3] and (0 <= b[0] - w[2] <= 15 or 0 <= w[0] - b[2] <= 15):
            return True
    return False


def _cluster(boxes: list[Box], gap: float) -> list[Box]:
    clusters = [b for b in boxes]
    changed = True
    while changed:
        changed = False
        out: list[Box] = []
        for b in clusters:
            for i, c in enumerate(out):
                if area((max(b[0], c[0] - gap), max(b[1], c[1] - gap),
                         min(b[2], c[2] + gap), min(b[3], c[3] + gap))) > 0:
                    out[i] = union(b, c)
                    changed = True
                    break
            else:
                out.append(b)
        clusters = out
    return clusters


def _table_boxes(page: pymupdf.Page) -> list[Box]:
    try:
        return [tuple(t.bbox) for t in page.find_tables().tables]
    except Exception:          # find_tables is best-effort
        return []


def _attach_labels(page: pymupdf.Page, box: Box, cfg: DetectConfig) -> Box:
    """Grow the box to include short text lines (labels) inside / adjacent to it.

    Long lines are question text and are never absorbed.
    """
    reach = expand(box, cfg.label_reach_pt)
    grown = box
    d = page.get_text("dict", clip=pymupdf.Rect(reach))
    for blk in d.get("blocks", []):
        for ln in blk.get("lines", []):
            txt = "".join(s["text"] for s in ln["spans"]).strip()
            lb = tuple(ln["bbox"])
            if not txt or len(txt) > cfg.label_max_chars:
                continue
            if containment(lb, reach) < 0.9:
                continue
            grown = union(grown, lb)
    return grown


def _is_text_only(page: pymupdf.Page, box: Box) -> bool:
    words = page.get_text("words", clip=pymupdf.Rect(box))
    wa = sum(area(tuple(w[:4])) for w in words)
    return wa > 0.45 * area(box)


def detect_page_native(page: pymupdf.Page, boiler: set, cfg: DetectConfig = DetectConfig()) -> tuple[list[dict], bool]:
    els, scanned = _graphic_elements(page, boiler, cfg)
    tables: list[Box] = []
    if cfg.skip_tables:
        lines = [e for e in els if _kind(e, cfg) in ("hline", "vline")]
        tables = table_regions(lines, page.rect.width, page.rect.height, cfg) + _table_boxes(page)
        els = [e for e in els if not any(containment(e, t) > 0.9 for t in tables)]
    # cluster, remembering which element kinds each cluster holds
    clusters = _cluster(els, cfg.merge_gap_pt)
    dets = []
    for c in clusters:
        members = [e for e in els if containment(e, c) > 0.99]
        kinds = {_kind(e, cfg) for e in members}
        has_image = any(containment(tuple(i["bbox"]), c) > 0.99 for i in page.get_image_info())
        big_side = max((max(e[2] - e[0], e[3] - e[1]) for e in members if _kind(e, cfg) != "hline"), default=0)
        c = _attach_labels(page, c, cfg)
        w, h = c[2] - c[0], c[3] - c[1]
        reason = ""
        if w < cfg.min_w_pt or h < cfg.min_h_pt or area(c) < cfg.min_area_pt2:
            reason = "too_small"
        elif kinds <= {"hline"} or (not has_image and big_side < 20):
            reason = "equation_or_rule"      # fraction bars, vector accents, symbols
        elif max(w, h) / max(min(w, h), 1e-6) > cfg.max_aspect:
            reason = "banner_or_strip"
        elif area(c) > cfg.max_page_frac * page.rect.width * page.rect.height:
            reason = "too_large"
        elif any(containment(c, t) > 0.8 for t in tables):
            reason = "inside_table"
        elif any(iou(c, t) > 0.7 for t in tables):
            reason = "is_table"
        elif _is_text_only(page, c):
            reason = "text_dominated"
        dets.append({"box_pt": c, "accepted": not reason, "reason": reason, "method": "native"})
    # merge clusters that grew into each other after label attachment
    acc = [d for d in dets if d["accepted"]]
    merged = _cluster([d["box_pt"] for d in acc], 0)
    rej = [d for d in dets if not d["accepted"]]
    return [{"box_pt": b, "accepted": True, "reason": "", "method": "native"} for b in merged] + rej, scanned


def detect_page_raster(page: pymupdf.Page, cfg: DetectConfig = DetectConfig(), dpi: int = 100) -> list[dict]:
    """Fallback for scanned pages: big non-text ink blobs."""
    img = render(page, dpi=dpi)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ink = (g < 170).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    keep = np.zeros_like(ink)
    text_h = 0.22 * dpi   # ~16pt glyph height
    for k in range(1, n):
        x, y, w, h, a = st[k]
        if h > text_h * 1.6 or w > text_h * 6:     # bigger than a glyph/word
            keep[lab == k] = 1
    keep = cv2.dilate(keep, np.ones((15, 15), np.uint8))
    n, _, st, _ = cv2.connectedComponentsWithStats(keep, 8)
    s = 72.0 / dpi
    H, W = g.shape
    out = []
    for k in range(1, n):
        x, y, w, h, _ = st[k]
        b = (x * s, y * s, (x + w) * s, (y + h) * s)
        reason = ""
        if w * s < cfg.min_w_pt or h * s < cfg.min_h_pt:
            reason = "too_small"
        elif w > 0.9 * W and h < 0.02 * H:
            reason = "rule"
        elif w * h > cfg.max_page_frac * W * H:
            reason = "too_large"
        out.append({"box_pt": b, "accepted": not reason, "reason": reason, "method": "raster"})
    return out


def is_scanned_page(page: pymupdf.Page, cfg: DetectConfig = DetectConfig()) -> bool:
    pa = page.rect.width * page.rect.height
    big = any(area(tuple(i["bbox"])) >= cfg.max_page_frac * pa for i in page.get_image_info())
    return big and len(page.get_text("words")) < 20


def scanned_fraction(doc: pymupdf.Document, cfg: DetectConfig = DetectConfig()) -> float:
    return sum(is_scanned_page(p, cfg) for p in doc) / max(len(doc), 1)


def detect_page(page: pymupdf.Page, boiler: set, cfg: DetectConfig = DetectConfig(),
                doc_scanned_frac: float = 0.0) -> list[dict]:
    """doc_scanned_frac: share of scanned pages in this document. Isolated scanned
    pages in a digital document are covers/instructions and are skipped."""
    dets, _ = detect_page_native(page, boiler, cfg)
    if (cfg.raster_fallback and doc_scanned_frac >= cfg.scanned_doc_frac
            and is_scanned_page(page, cfg) and not any(d["accepted"] for d in dets)):
        return detect_page_raster(page, cfg)
    return dets

In [ ]:
%%writefile /content/figx/pipeline.py
"""End-to-end pipeline: inventory -> ground truth -> evaluation -> bulk extraction.

All state is in JSONL manifests under <out>/manifests so a disconnected Colab
session resumes where it stopped. Crop ids are deterministic
(<pdf sha1[:10]>_p<page>_<gt|fig><n>), so reruns overwrite rather than duplicate.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import logging
import re
import time
import traceback
from dataclasses import asdict
from pathlib import Path

import cv2
import numpy as np
import pymupdf

from .core import (RedConfig, area, containment, crop_region, iou, map_marked_to_original,
                   quality_flags, recover_annotations, render, save_png)
from .detect import DetectConfig, boilerplate_keys, detect_page, scanned_fraction

log = logging.getLogger("figx")
MARK_RE = re.compile(r"\s*red\s*box(es)?\s*marked", re.I)
CROP_DPI = 300


# ------------------------------------------------------------------ utils
def sha1(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding="utf8").splitlines() if l.strip()]


def append_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def _r(b) -> list[float]:
    return [round(float(v), 2) for v in b]


def setup_logging(out: Path) -> None:
    (out / "logs").mkdir(parents=True, exist_ok=True)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    log.setLevel(logging.INFO)
    if not log.handlers:
        for name, lvl in (("extraction.log", logging.INFO), ("errors.log", logging.ERROR)):
            h = logging.FileHandler(out / "logs" / name, encoding="utf8")
            h.setLevel(lvl)
            h.setFormatter(fmt)
            log.addHandler(h)
        sh = logging.StreamHandler()
        sh.setFormatter(fmt)
        log.addHandler(sh)


# -------------------------------------------------------------- inventory
def inventory(roots: list[Path], out: Path) -> list[dict]:
    """Discover PDFs, hash them, flag duplicates, marked/original pairs, scans."""
    rows, seen = [], {}
    pdfs = sorted({p.resolve() for r in roots for p in Path(r).rglob("*") if p.suffix.lower() == ".pdf"})
    for p in pdfs:
        row = {"path": str(p), "name": p.name, "exam": next((x for x in ("JEE", "NEET") if x in p.parts), "OTHER"),
               "bytes": p.stat().st_size, "marked": bool(MARK_RE.search(p.stem))}
        try:
            row["sha1"] = sha1(p)
            with pymupdf.open(p) as d:
                if d.needs_pass:
                    raise ValueError("encrypted")
                row.update(pages=len(d), scanned_frac=round(scanned_fraction(d), 2),
                           page_size_pt=_r(d[0].rect[2:]) if len(d) else None)
            row["duplicate_of"] = seen.get(row["sha1"])
            seen.setdefault(row["sha1"], str(p))
            row["status"] = "ok"
        except Exception as e:  # corrupt / encrypted files are reported, not fatal
            row.update(status="error", error=f"{type(e).__name__}: {e}")
        rows.append(row)
    by_stem = {Path(r["path"]).stem.lower(): r for r in rows}
    for r in rows:
        if r["marked"]:
            orig = by_stem.get(MARK_RE.sub("", Path(r["path"]).stem).lower())
            r["original_path"] = orig["path"] if orig else None
    (out / "manifests").mkdir(parents=True, exist_ok=True)
    (out / "manifests" / "input_pdfs.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf8")
    return rows


# ----------------------------------------------------------- ground truth
def run_ground_truth(marked_path: Path, original_path: Path | None, out: Path,
                     cfg: RedConfig = RedConfig(), dpi: int = CROP_DPI) -> list[dict]:
    """Recover human boxes from a marked PDF; crop clean figures from the original."""
    marked_path = Path(marked_path)
    original_path = Path(original_path) if original_path else None
    src_sha = sha1(original_path or marked_path)
    stem = (original_path or marked_path).stem
    rows: list[dict] = []
    with pymupdf.open(marked_path) as M, (pymupdf.open(original_path) if original_path else pymupdf.open()) as O:
        if original_path and len(M) != len(O):
            raise ValueError(f"page count mismatch {len(M)} vs {len(O)}")
        for i in range(len(M)):
            orig_page = O[i] if original_path else None
            anns, diag = recover_annotations(M[i], orig_page, cfg)
            n = 0
            for a in anns:
                base = {"source_pdf": str(original_path or marked_path), "marked_pdf": str(marked_path),
                        "source_sha1": src_sha, "page": i + 1, "method": f"gt_{a.method}",
                        "marked_box_pt": _r(a.box_pt), "frame_thickness_pt": round(a.thickness_pt, 2),
                        "frame_check": a.check, "align_shift_px": diag.get("shift_px")}
                if not a.accepted:
                    rows.append({**base, "status": "rejected", "reason": a.reason})
                    continue
                n += 1
                crop_id = f"{src_sha[:10]}_p{i + 1:03d}_gt{n:02d}"
                page = orig_page or M[i]
                inner = map_marked_to_original(a.inner_pt, M[i], page, diag.get("shift_px") or (0, 0),
                                               cfg.analysis_dpi) if orig_page else a.inner_pt
                img, tight = crop_region(page, inner, dpi=dpi, tight=True)
                q = quality_flags(img, page, tight, cfg)
                path = out / "ground_truth" / "crops" / stem / f"{crop_id}.png"
                save_png(img, path, dpi)
                rows.append({**base, "status": "accepted", "crop_id": crop_id, "crop_path": str(path),
                             "box_inner_pt": _r(inner), "box_tight_pt": _r(tight),
                             "page_size_pt": _r(page.rect[2:]), "crop_px": [img.shape[1], img.shape[0]],
                             "dpi": dpi, "quality": q, "review_status": "needs_review" if q["flags"] else "auto_ok"})
    mf = out / "manifests" / "ground_truth.jsonl"
    keep = [r for r in read_jsonl(mf) if r.get("marked_pdf") != str(marked_path)]   # idempotent rerun
    mf.parent.mkdir(parents=True, exist_ok=True)
    mf.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in keep + rows), encoding="utf8")
    return rows


# --------------------------------------------------------------- detector
def extract_pdf(pdf: Path, out: Path, dcfg: DetectConfig = DetectConfig(), dpi: int = CROP_DPI,
                pages: list[int] | None = None, subdir: str = "figures") -> tuple[list[dict], dict]:
    pdf = Path(pdf)
    s = sha1(pdf)
    rows, stats = [], {"pages": 0, "candidates": 0, "accepted": 0, "rejected": 0}
    t0 = time.time()
    with pymupdf.open(pdf) as doc:
        boiler = boilerplate_keys(doc, dcfg)
        sf = scanned_fraction(doc, dcfg)
        for i in pages or range(len(doc)):
            page = doc[i]
            stats["pages"] += 1
            try:
                dets = detect_page(page, boiler, dcfg, sf)
            except Exception as e:
                log.error("page failed %s p%d: %s", pdf.name, i + 1, e)
                rows.append({"source_pdf": str(pdf), "source_sha1": s, "page": i + 1, "status": "error",
                             "reason": f"{type(e).__name__}: {e}"})
                continue
            n = 0
            for d in dets:
                stats["candidates"] += 1
                base = {"source_pdf": str(pdf), "source_sha1": s, "page": i + 1, "method": d["method"],
                        "candidate_box_pt": _r(d["box_pt"])}
                if not d["accepted"]:
                    stats["rejected"] += 1
                    rows.append({**base, "status": "rejected", "reason": d["reason"]})
                    continue
                n += 1
                crop_id = f"{s[:10]}_p{i + 1:03d}_fig{n:02d}"
                img, tight = crop_region(page, d["box_pt"], dpi=dpi, tight=True)
                q = quality_flags(img, page, tight)
                if d["method"] == "raster":
                    q["flags"].append("raster_fallback")
                path = out / subdir / pdf.stem / f"{crop_id}.png"
                save_png(img, path, dpi)
                stats["accepted"] += 1
                rows.append({**base, "status": "accepted", "crop_id": crop_id, "crop_path": str(path),
                             "box_tight_pt": _r(tight), "page_size_pt": _r(page.rect[2:]),
                             "crop_px": [img.shape[1], img.shape[0]], "dpi": dpi, "quality": q,
                             "review_status": "needs_review" if q["flags"] else "auto_ok"})
    stats["seconds"] = round(time.time() - t0, 2)
    return rows, stats


def run_bulk(pdfs: list[Path], out: Path, dcfg: DetectConfig = DetectConfig(), dpi: int = CROP_DPI,
             limit: int | None = None) -> dict:
    """Resumable bulk extraction. Completed PDFs (by sha1) are skipped on rerun."""
    setup_logging(out)
    mdir = out / "manifests"
    done = {r["sha1"] for r in read_jsonl(mdir / "completed_pdfs.jsonl")}
    todo = []
    for p in pdfs:
        h = sha1(Path(p))
        if h in done or h in {x[1] for x in todo}:
            continue
        todo.append((Path(p), h))
    todo = todo[:limit] if limit else todo
    log.info("bulk: %d to process, %d already done", len(todo), len(done))
    summary = {"processed": 0, "failed": 0, "accepted": 0}
    for k, (p, h) in enumerate(todo, 1):
        try:
            rows, st = extract_pdf(p, out, dcfg, dpi)
            append_jsonl(mdir / "crop_manifest.jsonl", rows)
            append_jsonl(mdir / "completed_pdfs.jsonl", [{"sha1": h, "path": str(p), **st}])
            summary["processed"] += 1
            summary["accepted"] += st["accepted"]
            log.info("[%d/%d] %s: %d pages, %d crops, %.1fs", k, len(todo), p.name, st["pages"], st["accepted"], st["seconds"])
        except Exception as e:     # one bad PDF never stops the run
            summary["failed"] += 1
            log.error("FAILED %s: %s", p, e)
            append_jsonl(mdir / "failed_files.jsonl", [{"path": str(p), "sha1": h, "error": f"{type(e).__name__}: {e}",
                                                       "trace": traceback.format_exc()[-1500:]}])
    return summary


# ------------------------------------------------------------- evaluation
def evaluate(gt_rows: list[dict], det_rows: list[dict], iou_thr: float = 0.5) -> dict:
    """Match detector crops to ground-truth crops (both as tight content boxes).

    Reports precision/recall at IoU>=iou_thr plus completeness (share of the GT
    content box covered by the matched detection) and contamination (share of
    the detection lying outside the GT box).
    """
    gts = [r for r in gt_rows if r["status"] == "accepted"]
    dets = [r for r in det_rows if r["status"] == "accepted"]
    pages = sorted({(r["page"]) for r in gts} | {r["page"] for r in dets})
    tp, matches, fps, fns = 0, [], [], []
    for pg in pages:
        G = [g for g in gts if g["page"] == pg]
        D = [d for d in dets if d["page"] == pg]
        pairs = sorted(((iou(tuple(d["box_tight_pt"]), tuple(g["box_tight_pt"])), di, gi)
                        for di, d in enumerate(D) for gi, g in enumerate(G)), reverse=True)
        ud, ug = set(), set()
        for v, di, gi in pairs:
            if v < iou_thr or di in ud or gi in ug:
                continue
            ud.add(di); ug.add(gi); tp += 1
            d, g = tuple(D[di]["box_tight_pt"]), tuple(G[gi]["box_tight_pt"])
            matches.append({"page": pg, "iou": round(v, 3), "completeness": round(containment(g, d), 3),
                            "contamination": round(1 - containment(d, g), 3),
                            "gt": G[gi]["crop_id"], "det": D[di]["crop_id"]})
        fps += [{"page": pg, "det": D[i]["crop_id"], "box": D[i]["box_tight_pt"]} for i in range(len(D)) if i not in ud]
        fns += [{"page": pg, "gt": G[i]["crop_id"], "box": G[i]["box_tight_pt"]} for i in range(len(G)) if i not in ug]
    gt_pages = {g["page"] for g in gts}
    neg_fp = [f for f in fps if f["page"] not in gt_pages]
    m = lambda k: round(float(np.mean([x[k] for x in matches])), 3) if matches else None
    return {"gt": len(gts), "det": len(dets), "tp": tp, "fp": len(fps), "fn": len(fns),
            "precision": round(tp / max(len(dets), 1), 3), "recall": round(tp / max(len(gts), 1), 3),
            "mean_iou": m("iou"), "mean_completeness": m("completeness"), "mean_contamination": m("contamination"),
            "fp_on_pages_without_gt": len(neg_fp), "matches": matches, "false_positives": fps, "misses": fns}


# ---------------------------------------------------------------- preview
def contact_sheet(rows: list[dict], path: Path, thumb_h: int = 160, cols: int = 5, max_items: int = 60) -> Path | None:
    items = [r for r in rows if r.get("status") == "accepted"][:max_items]
    if not items:
        return None
    tiles = []
    for r in items:
        im = cv2.imread(r["crop_path"])
        if im is None:
            continue
        s = thumb_h / im.shape[0]
        im = cv2.resize(im, (max(1, min(int(im.shape[1] * s), thumb_h * 3)), thumb_h), interpolation=cv2.INTER_AREA)
        flags = ",".join(r["quality"]["flags"]) if r.get("quality") else ""
        lab = f"p{r['page']} {r['crop_id'][-5:]} {flags}"[:48]
        im = cv2.copyMakeBorder(im, 22, 6, 6, 6, cv2.BORDER_CONSTANT,
                                value=(170, 170, 255) if flags else (200, 230, 200))
        cv2.putText(im, lab, (4, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1)
        tiles.append(im)
    w = max(t.shape[1] for t in tiles)
    tiles = [cv2.copyMakeBorder(t, 0, 0, 0, w - t.shape[1], cv2.BORDER_CONSTANT, value=(255, 255, 255)) for t in tiles]
    while len(tiles) % cols:
        tiles.append(np.full_like(tiles[0], 255))
    sheet = np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])
    save_png(sheet, Path(path), 72)
    return Path(path)


# ------------------------------------------------------------ YOLO export
def export_yolo(gt_rows: list[dict], out: Path, dpi: int = 150, val_pages: set | None = None,
                negatives_per_pdf: int = 5, seed: int = 0) -> Path:
    """Page images + normalized labels from REVIEWED/accepted ground truth only.

    Also exports a few pages without targets as negatives. Split is by page here
    because only one marked PDF exists; with more marked PDFs split by document.
    """
    rng = np.random.default_rng(seed)
    acc = [r for r in gt_rows if r["status"] == "accepted" and r.get("review_status") != "rejected"]
    root = out / "dataset_yolo"
    by_pdf: dict = {}
    for r in acc:
        by_pdf.setdefault(r["source_pdf"], []).append(r)
    for pdf, rs in by_pdf.items():
        with pymupdf.open(pdf) as doc:
            pos = sorted({r["page"] for r in rs})
            negs = [p for p in range(1, len(doc) + 1) if p not in pos]
            negs = list(rng.choice(negs, size=min(negatives_per_pdf, len(negs)), replace=False)) if negs else []
            for pg in pos + [int(n) for n in negs]:
                split = "val" if val_pages and pg in val_pages else "train"
                page = doc[pg - 1]
                img = render(page, dpi=dpi)
                stem = f"{Path(pdf).stem.replace(' ', '_')}_p{pg:03d}"
                save_png(img, root / "images" / split / f"{stem}.png", dpi)
                W, H = page.rect.width, page.rect.height
                lines = []
                for r in (x for x in rs if x["page"] == pg):
                    x0, y0, x1, y1 = r["box_tight_pt"]
                    cx, cy, bw, bh = (x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H
                    assert 0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1, r["crop_id"]
                    lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                lp = root / "labels" / split / f"{stem}.txt"
                lp.parent.mkdir(parents=True, exist_ok=True)
                lp.write_text("\n".join(lines) + ("\n" if lines else ""))
    (root / "data.yaml").write_text(f"path: {root.as_posix()}\ntrain: images/train\nval: images/val\nnames: ['figure']\n")
    return root


# -------------------------------------------------------------------- CLI
def main(argv=None):
    ap = argparse.ArgumentParser(description="JEE/NEET figure extraction")
    sub = ap.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("inventory"); a.add_argument("roots", nargs="+"); a.add_argument("--out", required=True)
    g = sub.add_parser("groundtruth"); g.add_argument("marked"); g.add_argument("--original"); g.add_argument("--out", required=True)
    b = sub.add_parser("bulk"); b.add_argument("roots", nargs="+"); b.add_argument("--out", required=True); b.add_argument("--limit", type=int)
    e = sub.add_parser("evaluate"); e.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    out = Path(args.out)
    setup_logging(out)
    if args.cmd == "inventory":
        rows = inventory([Path(r) for r in args.roots], out)
        print(json.dumps({"pdfs": len(rows), "unique": len({r.get('sha1') for r in rows}),
                          "marked": sum(r["marked"] for r in rows)}, indent=1))
    elif args.cmd == "groundtruth":
        rows = run_ground_truth(Path(args.marked), Path(args.original) if args.original else None, out)
        print(sum(r["status"] == "accepted" for r in rows), "accepted;", sum(r["status"] == "rejected" for r in rows), "rejected")
    elif args.cmd == "bulk":
        pdfs = [p for r in args.roots for p in Path(r).rglob("*.pdf") if not MARK_RE.search(p.stem)]
        print(run_bulk(sorted(pdfs), out, limit=args.limit))
    elif args.cmd == "evaluate":
        gt = read_jsonl(out / "manifests" / "ground_truth.jsonl")
        res = {}
        for src in sorted({r["source_pdf"] for r in gt}):
            det, _ = extract_pdf(Path(src), out, subdir="eval_detections")
            res[src] = evaluate([r for r in gt if r["source_pdf"] == src], det)
        (out / "reports").mkdir(parents=True, exist_ok=True)
        (out / "reports" / "evaluation.json").write_text(json.dumps(res, indent=1))
        for k, v in res.items():
            print(Path(k).name, {x: v[x] for x in ("gt", "det", "tp", "fp", "fn", "precision", "recall", "mean_iou",
                                                    "mean_completeness", "mean_contamination", "fp_on_pages_without_gt")})


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
from google.colab import drive
MOUNT = Path('/content/drive')
if not (MOUNT/'MyDrive').exists():
    drive.mount(str(MOUNT))          # mounts once; skipped if already mounted
assert (MOUNT/'MyDrive').exists(), 'Drive not mounted'
sys.path.insert(0, '/content')

## 1. Configuration (edit only this cell)

In [ ]:
DRIVE = MOUNT/'MyDrive'
REF_DIR   = DRIVE/'Reference_PDFs'                      # holds the original + red-boxes-marked PDFs
ORIG_PDF  = REF_DIR/'Copy of original (4).pdf'
MARK_PDF  = REF_DIR/'Copy of original (4) red boxes marked.pdf'
PDF_ROOTS = [DRIVE/'JEE', DRIVE/'NEET']                  # adjust: shortcuts to the shared dataset folders must live in My Drive
OUT       = DRIVE/'JEE_NEET_Extracted_Images'/'v4_figx'  # NEW folder; old stage2_* outputs untouched
LIMIT     = None                                         # set e.g. 5 for a trial bulk run
for p in (ORIG_PDF, MARK_PDF): print(p.exists(), p)
for r in PDF_ROOTS: print(r.exists(), r, len(list(r.rglob('*.pdf'))) if r.exists() else 0, 'pdfs')

## 2. Reference verification + inventory
Checks page counts, geometry, duplicates (by sha1), marked/original pairing, scans, corrupt files.

In [ ]:
from figx.pipeline import *
import pymupdf, json
assert ORIG_PDF.exists() and MARK_PDF.exists(), 'Put both reference PDFs in REF_DIR'
with pymupdf.open(ORIG_PDF) as O, pymupdf.open(MARK_PDF) as M:
    print('pages', len(O), len(M), '| page0', O[0].rect, M[0].rect)
rows = inventory([r for r in PDF_ROOTS if r.exists()] + [REF_DIR], OUT)
import collections
print(collections.Counter((r['exam'], r['status']) for r in rows), '| unique:', len({r.get('sha1') for r in rows}))

## 3. Ground truth from the marked PDF
Diff-based box recovery (marked vs original), clean crops rendered from the ORIGINAL at 300 dpi.

In [ ]:
gt = run_ground_truth(MARK_PDF, ORIG_PDF, OUT)
print(sum(r['status']=='accepted' for r in gt), 'accepted,', sum(r['status']=='rejected' for r in gt), 'rejected')

## 4. Visual review (no reprocessing needed)

In [ ]:
from IPython.display import Image, display
display(Image(str(contact_sheet(gt, OUT/'reports'/'gt_preview.png', thumb_h=130, cols=4))))

## 5. Evaluate the unmarked-PDF detector against the human boxes
⚠ Only one marked PDF exists, and the detector rules were tuned on it, so this is a **development** score, not an independent one. Mark 3–5 more PDFs (different layouts) and add them to `ground_truth` for a real held-out test.

In [ ]:
det, st = extract_pdf(ORIG_PDF, OUT, subdir='eval_detections')
res = evaluate(gt, det)
print({k: res[k] for k in ('gt','det','tp','fp','fn','precision','recall','mean_iou','mean_completeness','mean_contamination','fp_on_pages_without_gt')}, st)
display(Image(str(contact_sheet(det, OUT/'reports'/'det_preview.png', thumb_h=130, cols=4))))

## 6. Bulk processing of unmarked PDFs (resumable)
Run with `LIMIT=5` first and inspect the preview. Re-running skips PDFs already in `manifests/completed_pdfs.jsonl`.

In [ ]:
pdfs = sorted(p for r in PDF_ROOTS if r.exists() for p in r.rglob('*.pdf') if not MARK_RE.search(p.stem))
summary = run_bulk(pdfs, OUT, limit=LIMIT)
print(summary)
cm = read_jsonl(OUT/'manifests'/'crop_manifest.jsonl')
print('accepted', sum(r['status']=='accepted' for r in cm), '| needs_review', sum(r.get('review_status')=='needs_review' for r in cm))
sheet = contact_sheet(cm, OUT/'reports'/'bulk_preview.png', thumb_h=110, cols=6, max_items=120)
sheet and display(Image(str(sheet)))

## 7. Export
CSV manifest + YOLO labels (from accepted ground truth only).

In [ ]:
import csv
cm = [r for r in read_jsonl(OUT/'manifests'/'crop_manifest.jsonl') if r['status']=='accepted']
with open(OUT/'manifests'/'crop_manifest.csv','w',newline='',encoding='utf8') as f:
    w = csv.writer(f); w.writerow(['crop_id','source_pdf','page','box_tight_pt','crop_px','method','review_status','flags','crop_path'])
    for r in cm: w.writerow([r['crop_id'], r['source_pdf'], r['page'], r['box_tight_pt'], r['crop_px'], r['method'], r['review_status'], ';'.join(r['quality']['flags']), r['crop_path']])
print(export_yolo(read_jsonl(OUT/'manifests'/'ground_truth.jsonl'), OUT))